# HID Open-Set Analysis: mask ratio / strategy / patch size

Standalone notebook, separate from `consolidated_visualization_hid.ipynb`'s closed-set sections and from the (deleted) `build_best_layer_table` function used for HAR.

**Why a separate file, not a shared helper with HAR:** HAR's closed-set data (`knn_acc`/`lp_acc`/`mlp_acc`) and HID's open-set data (`rank1_acc`/`verif_auc`/`verif_eer`) have fundamentally different shapes. A function that tries to find the "best" result by selecting across *different metric types* (e.g. picking whichever of KNN/LP/MLP happens to be highest at a given layer) silently introduces selection bias -- the reported number becomes an oracle upper bound, not a number any single deployed pipeline would actually produce. This was traced as the likely cause of an anomalously high "MAE + best layer" result in the HAR analysis.

**This notebook's rule:** when selecting a "best layer" for a config, we select using **one fixed metric only** (`verif_auc`), never across different metric types. This is a much weaker, more defensible form of selection (still worth flagging as best-layer-selected in the paper's Methods section), not a cross-metric oracle.

**`test_cross_device`** contains 6 known identities + 1 novel identity (`U02`) -- a real, non-degenerate open-set comparison across configs.

**`test_cross_user`** contains ONLY the novel identity `U02`. Rank-1 retrieval is trivially ~100% regardless of embedding quality (every gallery member necessarily shares the query's identity), and verification AUC/EER are undefined (no different-identity pairs exist). This notebook reports `test_cross_user` as a labeled, flagged summary table -- **never as a bar chart compared across configs**, since doing so would visually imply a meaningful comparison that doesn't exist.

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
RESULTS_DIR = Path("/home/zhuzih19/csi-project/csi-fall-detection/results/mae_hid")
FIG_DIR = Path("/home/zhuzih19/csi-project/csi-fall-detection/figs/mae_hid_openset")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Shared loading utilities (same pattern as the closed-set HID notebook)

In [ ]:
def load_matching_results(results_dir, filters, debug=False):
    """Scans results_dir for *.json files whose saved args match every key in `filters`."""
    matched = []
    for path in sorted(results_dir.glob("*.json")):
        try:
            with open(path) as f:
                result = json.load(f)
        except (json.JSONDecodeError, UnicodeDecodeError):
            if debug:
                print(f"[skip: unreadable] {path.name}")
            continue
        args = result.get('args', {})
        mismatch = None
        for key, val in filters.items():
            if args.get(key) != val:
                mismatch = f"{key}={args.get(key)!r} != {val!r}"
                break
        if mismatch:
            if debug:
                print(f"[skip: {mismatch}] {path.name}")
            continue
        if debug:
            print(f"[match] {path.name}")
        matched.append(result)
    return matched


def group_by(results, key_fn):
    """Groups a list of result dicts by key_fn(result['args'])."""
    groups = defaultdict(list)
    for r in results:
        try:
            key = key_fn(r)
        except (KeyError, TypeError):
            continue
        groups[key].append(r)
    return dict(groups)


def get_checkpoints(result):
    return sorted(result['evals'].keys(), key=lambda k: int(k.split('_')[1]))


def get_layers(result, ckpt=None):
    ckpt = ckpt or get_checkpoints(result)[-1]
    return sorted(result['evals'][ckpt].keys(), key=lambda k: int(k.split('_')[1]))

## Best-layer selection -- single metric only, no cross-protocol selection

`best_layer_openset_metric` selects the layer (at the final checkpoint) that maximizes a **single, fixed** metric (default `verif_auc`) for one run. It never compares across `rank1_acc` / `verif_auc` / `verif_eer` to pick whichever looks best -- that cross-metric selection is exactly the pattern suspected of inflating the HAR "best layer" numbers.

In [ ]:
def best_layer_openset_metric(result, split, metric='verif_auc'):
    """Returns (best_layer_int, best_value) for `metric` on `split`, at the final checkpoint,
    selecting ONLY over layer (never over a different metric). Returns (None, None) if the
    metric is undefined (None) at every layer -- e.g. verif_auc on a single-identity split."""
    final_ckpt = get_checkpoints(result)[-1]
    layers = get_layers(result, final_ckpt)
    best_layer, best_val = None, None
    for layer_key in layers:
        d = result['evals'][final_ckpt][layer_key].get(split)
        if not d or d.get(metric) is None:
            continue
        val = d[metric]
        if best_val is None or val > best_val:
            best_val = val
            best_layer = int(layer_key.split('_')[1])
    return best_layer, best_val


def build_openset_table(groups, split, metric, group_label_fmt="{k}"):
    """For each group (e.g. each mask ratio), returns mean/std of `metric` on `split`,
    where each run's contribution is ITS OWN best-layer value for that same metric
    (single-metric selection per run, then averaged across seeds)."""
    rows = []
    for key, runs in sorted(groups.items(), key=lambda kv: (isinstance(kv[0], str), kv[0])):
        vals = []
        layers_used = []
        for r in runs:
            layer, val = best_layer_openset_metric(r, split, metric)
            if val is not None:
                vals.append(val)
                layers_used.append(layer)
        if not vals:
            rows.append({'group': group_label_fmt.format(k=key), 'mean': None, 'std': None,
                        'n': 0, 'layers_used': []})
            continue
        rows.append({
            'group': group_label_fmt.format(k=key),
            'mean': float(np.mean(vals)),
            'std': float(np.std(vals)) if len(vals) > 1 else 0.0,
            'n': len(vals),
            'layers_used': layers_used,
        })
    return pd.DataFrame(rows)

## Plotting: `test_cross_device` only (real, non-degenerate open-set comparison)

In [ ]:
def plot_cross_device_openset(groups, axis_title, filename, metric='verif_auc'):
    """Bar chart of test_cross_device's `metric`, one bar per group, using each run's own
    best-layer value for that metric (single-metric selection, see build_openset_table)."""
    df = build_openset_table(groups, 'test_cross_device', metric)
    df_valid = df[df['mean'].notna()]
    if df_valid.empty:
        print(f"[warn] no valid '{metric}' values found for test_cross_device across {axis_title}")
        return df

    fig, ax = plt.subplots(figsize=(9, 5.5))
    x = np.arange(len(df_valid))
    bars = ax.bar(x, df_valid['mean'], yerr=df_valid['std'], capsize=4,
                  color='#0891B2', alpha=0.85)
    ax.bar_label(bars, labels=[f"{v:.3f}\n(n={n})" for v, n in zip(df_valid['mean'], df_valid['n'])],
                fontsize=8, padding=3)
    ax.set_xticks(x)
    ax.set_xticklabels(df_valid['group'])
    ax.set_ylabel(metric)
    if metric == 'verif_auc':
        ax.set_ylim(0.4, 1.0)
        ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='chance (AUC=0.5)')
        ax.legend(fontsize=8)
    ax.set_title(f"test_cross_device {metric} by {axis_title}\n"
                f"(best layer selected per-run using {metric} only -- no cross-metric selection)")
    ax.grid(alpha=0.3, axis='y')
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches='tight')
    plt.show()
    return df

## `test_cross_user`: flagged trivial summary -- printed table, NOT a bar chart

This split is 100% the single novel identity. Rank-1 is trivially ~1.0 for any embedding; verification AUC/EER are undefined. This is printed for transparency, not plotted as a comparison.

In [ ]:
def print_cross_user_trivial_summary(groups, axis_title):
    print(f"--- test_cross_user (single-identity split, TRIVIAL -- not a meaningful comparison) ---")
    print(f"    Axis: {axis_title}")
    for key, runs in sorted(groups.items(), key=lambda kv: (isinstance(kv[0], str), kv[0])):
        rank1_vals = []
        n_identities_seen = set()
        for r in runs:
            final_ckpt = get_checkpoints(r)[-1]
            layers = get_layers(r, final_ckpt)
            for layer_key in layers:
                d = r['evals'][final_ckpt][layer_key].get('test_cross_user')
                if d and d.get('rank1_acc') is not None:
                    rank1_vals.append(d['rank1_acc'])
                    n_identities_seen.add(d.get('n_identities'))
        if rank1_vals:
            print(f"  {key}: Rank1={np.mean(rank1_vals)*100:.1f}% (n_identities={sorted(n_identities_seen)}) "
                 f"-- uninformative by construction, VerifAUC/EER undefined")
        else:
            print(f"  {key}: no test_cross_user data found")
    print()

## Section A -- Mask Ratio (enc12)

In [ ]:
RATIO_FILTERS = {
    "mask_strategy": "random",
    "encoder_depth": 12,
    "patch_h": 29,
    "patch_w": 25,
    "epochs": 150,
}
DEBUG = False

ratio_results = load_matching_results(RESULTS_DIR, RATIO_FILTERS, debug=DEBUG)
if not ratio_results:
    print("No mask-ratio files matched. Set DEBUG=True and re-run.")
    ratio_groups = {}
else:
    ratio_groups = group_by(ratio_results, key_fn=lambda r: r["args"]["mask_ratio"])
    print(f"Found {len(ratio_results)} files across {len(ratio_groups)} mask ratios: "
         f"{ {k: len(v) for k, v in ratio_groups.items()} }")

In [ ]:
if ratio_groups:
    ratio_auc_df = plot_cross_device_openset(ratio_groups, "Mask Ratio", "hid_ratio_cross_device_verif_auc.png")
    print(ratio_auc_df.to_string(index=False))
    print()
    print_cross_user_trivial_summary(ratio_groups, "Mask Ratio")

## Section B -- Mask Strategy (enc12, mask_ratio=0.75)

In [ ]:
STRATEGY_FILTERS = {
    "mask_ratio": 0.75,
    "encoder_depth": 12,
    "patch_h": 29,
    "patch_w": 25,
    "epochs": 150,
}
DEBUG = False

strategy_results = load_matching_results(RESULTS_DIR, STRATEGY_FILTERS, debug=DEBUG)
if not strategy_results:
    print("No mask-strategy files matched. Set DEBUG=True and re-run.")
    strategy_groups = {}
else:
    strategy_groups = group_by(strategy_results, key_fn=lambda r: r["args"]["mask_strategy"])
    print(f"Found {len(strategy_results)} files across {len(strategy_groups)} strategies: "
         f"{ {k: len(v) for k, v in strategy_groups.items()} }")

In [ ]:
if strategy_groups:
    strategy_auc_df = plot_cross_device_openset(strategy_groups, "Mask Strategy", "hid_strategy_cross_device_verif_auc.png")
    print(strategy_auc_df.to_string(index=False))
    print()
    print_cross_user_trivial_summary(strategy_groups, "Mask Strategy")

## Section C -- Patch Size (enc12, mask_ratio=0.75, mask_strategy=random)

In [ ]:
PATCH_FILTERS = {
    "mask_ratio": 0.75,
    "mask_strategy": "random",
    "encoder_depth": 12,
    "epochs": 150,
}
DEBUG = False

patch_results = load_matching_results(RESULTS_DIR, PATCH_FILTERS, debug=DEBUG)
if not patch_results:
    print("No patch-size files matched. Set DEBUG=True and re-run.")
    patch_groups = {}
else:
    patch_groups = group_by(patch_results, key_fn=lambda r: r["args"]["patch_h"])
    print(f"Found {len(patch_results)} files across {len(patch_groups)} patch sizes: "
         f"{ {k: len(v) for k, v in patch_groups.items()} }")

In [ ]:
if patch_groups:
    patch_auc_df = plot_cross_device_openset(patch_groups, "Patch Size", "hid_patch_cross_device_verif_auc.png")
    print(patch_auc_df.to_string(index=False))
    print()
    print_cross_user_trivial_summary(patch_groups, "Patch Size")

## Notes / caveats

- **Best-layer selection here is single-metric only** (`verif_auc`, never mixed with `rank1_acc`/`verif_eer` or with HAR's `knn_acc`/`lp_acc`/`mlp_acc`). This avoids the cross-protocol selection bias suspected of inflating an earlier HAR "best layer" analysis. It is still a form of selection (best layer per run, on the eval set itself) -- report this explicitly in the paper's Methods section, not as a fixed-layer result.
- **`test_cross_user` is never plotted as a bar chart in this notebook.** It is printed as a labeled, flagged summary only, since the split's single-identity structure makes Rank-1 trivially ~100% and VerifAUC/EER undefined -- a bar chart here would visually imply a real comparison that does not exist.
- **`DEBUG=True`** in any filter cell prints per-file match/skip reasons -- use it if a group's `n` looks wrong.